# 08 — Optimisation post-deploiement

## Objectif

Analyser les performances du modele en production et les optimiser :
1. **Profiling** avec cProfile pour identifier les goulots d'etranglement
2. **Conversion ONNX** pour accelerer l'inference
3. **Benchmark** avant/apres avec validation de non-regression
4. **Documentation** des resultats

In [ ]:
import json
import time
import cProfile
import pstats
import io
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Charger le modele et les donnees
model = joblib.load('../model/model.pkl')
with open('../model/config.json') as f:
    config = json.load(f)

FEATURE_NAMES = config['feature_names']
THRESHOLD = config['threshold']

# Recreer le holdout set (meme split que notebook 04)
df_train = pd.read_csv('../data/train_preprocessed.csv')
df_train.columns = df_train.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)
y = df_train['TARGET']
X = df_train.drop(columns=['TARGET'])
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_test = X_test[FEATURE_NAMES]

print(f"Modele : {type(model).__name__}")
print(f"Holdout set : {X_test.shape}")
print(f"Features : {len(FEATURE_NAMES)}")

## 1. Profiling du modele LightGBM (baseline)

On utilise **cProfile** (module standard Python) pour identifier les fonctions 
les plus couteuses lors de l'inference.

In [ ]:
# --- Profiling : inference unitaire (1 client) ---
sample_single = X_test.iloc[[0]]

profiler = cProfile.Profile()
profiler.enable()
for _ in range(1000):
    model.predict_proba(sample_single)
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats('cumulative')
stats.print_stats(15)
print("=== Profiling : 1000 predictions unitaires ===")
print(stream.getvalue())

In [ ]:
# --- Benchmark baseline : mesures de latence ---
import tracemalloc

def benchmark_model(predict_fn, X_single, X_batch, n_single=1000, n_batch=100):
    """Mesure les performances d'inference."""
    results = {}
    
    # 1. Inference unitaire
    times_single = []
    for _ in range(n_single):
        start = time.perf_counter()
        predict_fn(X_single)
        times_single.append((time.perf_counter() - start) * 1000)
    
    results['single_mean_ms'] = np.mean(times_single)
    results['single_median_ms'] = np.median(times_single)
    results['single_p95_ms'] = np.percentile(times_single, 95)
    
    # 2. Inference batch (100 lignes)
    times_batch = []
    for _ in range(n_batch):
        start = time.perf_counter()
        predict_fn(X_batch)
        times_batch.append((time.perf_counter() - start) * 1000)
    
    results['batch_mean_ms'] = np.mean(times_batch)
    results['batch_median_ms'] = np.median(times_batch)
    
    # 3. Memoire
    tracemalloc.start()
    predict_fn(X_batch)
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    results['peak_memory_kb'] = peak_memory / 1024
    
    return results

# Benchmark LightGBM
X_single = X_test.iloc[[0]]
X_batch_100 = X_test.head(100)

baseline_results = benchmark_model(
    lambda x: model.predict_proba(x)[:, 1],
    X_single, X_batch_100
)

print("=== Benchmark LightGBM (baseline) ===")
print(f"Inference unitaire : {baseline_results['single_mean_ms']:.2f} ms (mean)")
print(f"                     {baseline_results['single_median_ms']:.2f} ms (median)")
print(f"                     {baseline_results['single_p95_ms']:.2f} ms (P95)")
print(f"Inference batch (100): {baseline_results['batch_mean_ms']:.2f} ms (mean)")
print(f"Memoire pic : {baseline_results['peak_memory_kb']:.0f} KB")
print(f"Taille modele : {os.path.getsize('../model/model.pkl') / 1024:.0f} KB")

import os
baseline_results['model_size_kb'] = os.path.getsize('../model/model.pkl') / 1024

## 2. Conversion du modele en ONNX

**ONNX** (Open Neural Network Exchange) est un format standardise qui permet 
d'optimiser l'inference via **ONNX Runtime**, un moteur d'execution haute performance.

Avantages attendus :
- Inference plus rapide (optimisations bas niveau, vectorisation)
- Format portable (independant du framework d'entrainement)
- Empreinte memoire reduite

In [ ]:
import onnxmltools
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

# Conversion LightGBM -> ONNX
n_features = len(FEATURE_NAMES)
initial_type = [('float_input', FloatTensorType([None, n_features]))]

onnx_model = onnxmltools.convert_lightgbm(
    model,
    initial_types=initial_type,
    target_opset=15
)

# Sauvegarder le modele ONNX
onnx_path = '../model/model_optimized.onnx'
onnxmltools.utils.save_model(onnx_model, onnx_path)

print(f"Modele ONNX sauvegarde : {onnx_path}")
print(f"Taille : {os.path.getsize(onnx_path) / 1024:.0f} KB")

# Charger avec ONNX Runtime
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name
print(f"\nONNX Runtime session creee")
print(f"Input name : {input_name}")
print(f"Input shape : {session.get_inputs()[0].shape}")

## 3. Validation de non-regression

Avant de deployer le modele ONNX, on **verifie que les predictions sont identiques** 
au modele original. Les differences doivent etre negligeables (< 1e-5) et l'AUC identique.

In [ ]:
# Predictions LightGBM (reference)
y_proba_lgbm = model.predict_proba(X_test)[:, 1]

# Predictions ONNX Runtime
X_test_float32 = X_test.values.astype(np.float32)
onnx_result = session.run(None, {input_name: X_test_float32})
y_proba_onnx = np.array([r[1] for r in onnx_result[1]])  # probabilite classe 1

# Comparaison
max_diff = np.max(np.abs(y_proba_lgbm - y_proba_onnx))
mean_diff = np.mean(np.abs(y_proba_lgbm - y_proba_onnx))
auc_lgbm = roc_auc_score(y_test, y_proba_lgbm)
auc_onnx = roc_auc_score(y_test, y_proba_onnx)

# Decisions avec le meme seuil
decisions_lgbm = (y_proba_lgbm >= THRESHOLD).astype(int)
decisions_onnx = (y_proba_onnx >= THRESHOLD).astype(int)
decisions_match = (decisions_lgbm == decisions_onnx).mean()

print("=== Validation de non-regression ===")
print(f"Max difference de probabilite   : {max_diff:.10f}")
print(f"Moyenne difference de probabilite: {mean_diff:.10f}")
print(f"AUC LightGBM : {auc_lgbm:.6f}")
print(f"AUC ONNX     : {auc_onnx:.6f}")
print(f"Decisions identiques : {decisions_match:.4%}")
print(f"\nConclusion : {'VALIDE - pas de regression' if max_diff < 1e-4 else 'ATTENTION - differences significatives'}")

## 4. Benchmark ONNX Runtime

In [ ]:
# Fonction de prediction ONNX
def onnx_predict(X):
    return session.run(None, {input_name: X.values.astype(np.float32)})[1]

X_single_float = X_test.iloc[[0]]
X_batch_float = X_test.head(100)

onnx_results = benchmark_model(
    lambda x: onnx_predict(x),
    X_single_float, X_batch_float
)

onnx_results['model_size_kb'] = os.path.getsize(onnx_path) / 1024

print("=== Benchmark ONNX Runtime ===")
print(f"Inference unitaire : {onnx_results['single_mean_ms']:.2f} ms (mean)")
print(f"                     {onnx_results['single_median_ms']:.2f} ms (median)")
print(f"                     {onnx_results['single_p95_ms']:.2f} ms (P95)")
print(f"Inference batch (100): {onnx_results['batch_mean_ms']:.2f} ms (mean)")
print(f"Memoire pic : {onnx_results['peak_memory_kb']:.0f} KB")
print(f"Taille modele : {onnx_results['model_size_kb']:.0f} KB")

## 5. Tableau recapitulatif des resultats

In [ ]:
# Tableau comparatif
comparison = pd.DataFrame({
    'Metrique': [
        'Inference unitaire (mean)',
        'Inference unitaire (median)',
        'Inference unitaire (P95)',
        'Inference batch 100 (mean)',
        'Memoire pic (KB)',
        'Taille modele (KB)',
        'AUC Holdout'
    ],
    'LightGBM (baseline)': [
        f"{baseline_results['single_mean_ms']:.2f} ms",
        f"{baseline_results['single_median_ms']:.2f} ms",
        f"{baseline_results['single_p95_ms']:.2f} ms",
        f"{baseline_results['batch_mean_ms']:.2f} ms",
        f"{baseline_results['peak_memory_kb']:.0f}",
        f"{baseline_results['model_size_kb']:.0f}",
        f"{auc_lgbm:.6f}"
    ],
    'ONNX Runtime': [
        f"{onnx_results['single_mean_ms']:.2f} ms",
        f"{onnx_results['single_median_ms']:.2f} ms",
        f"{onnx_results['single_p95_ms']:.2f} ms",
        f"{onnx_results['batch_mean_ms']:.2f} ms",
        f"{onnx_results['peak_memory_kb']:.0f}",
        f"{onnx_results['model_size_kb']:.0f}",
        f"{auc_onnx:.6f}"
    ],
    'Amelioration': [
        f"{(1 - onnx_results['single_mean_ms']/baseline_results['single_mean_ms'])*100:+.1f}%",
        f"{(1 - onnx_results['single_median_ms']/baseline_results['single_median_ms'])*100:+.1f}%",
        f"{(1 - onnx_results['single_p95_ms']/baseline_results['single_p95_ms'])*100:+.1f}%",
        f"{(1 - onnx_results['batch_mean_ms']/baseline_results['batch_mean_ms'])*100:+.1f}%",
        f"{(1 - onnx_results['peak_memory_kb']/baseline_results['peak_memory_kb'])*100:+.1f}%",
        f"{(1 - onnx_results['model_size_kb']/baseline_results['model_size_kb'])*100:+.1f}%",
        "identique"
    ]
})

print("=" * 80)
print("TABLEAU RECAPITULATIF : LightGBM vs ONNX Runtime")
print("=" * 80)
print(comparison.to_string(index=False))
print("=" * 80)

## 6. Conclusion et justification de la configuration finale

### Strategie d'optimisation retenue

La conversion en **ONNX Runtime** est la strategie d'optimisation principale :
- Elle ne modifie pas le modele lui-meme (pas de perte de precision)
- Elle exploite des optimisations bas niveau (vectorisation SIMD, parallelisme)
- Le format ONNX est portable et standardise

### Configuration finale deployee

| Composant | Choix | Justification |
|-----------|-------|---------------|
| **Modele** | LightGBM (ONNX) | Meilleur compromis AUC / cout metier |
| **Format** | ONNX Runtime | Inference plus rapide, format portable |
| **Seuil** | 0.54 | Optimise pour le cout metier (10xFN + FP) |
| **Framework API** | Gradio | Simple, UI integree, natif HF Spaces |
| **Conteneurisation** | Docker (Python 3.11 slim) | Leger, reproductible |
| **CI/CD** | GitHub Actions | Tests → Build Docker → Deploy HF Spaces |

### Deploiement de la version optimisee

Le modele ONNX est sauvegarde dans `model/model_optimized.onnx`. 
Pour l'activer dans l'API, il suffit de definir la variable d'environnement :

```bash
USE_ONNX=true python -m app.app
```

Le pipeline CI/CD deploie automatiquement la nouvelle version lors d'un push sur `main`.